# 🚀 GPT

In this notebook, we'll walk through the steps required to train your own GPT model on the wine review dataset

The code is adapted from the excellent [GPT tutorial](https://keras.io/examples/generative/text_generation_with_miniature_gpt/) created by Apoorv Nandan available on the Keras website.

In [55]:
# === Colab Setup (auto-skipped outside Google Colab) ===
import sys, os, subprocess, pathlib

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO = "yukai-yabuta/Generative_Deep_Learning_2nd_Edition"
    BRANCH = "colab-ch09"
    NOTEBOOK_REL = "notebooks/09_transformer/gpt"
    REPO_DIR = "/content/repo"

    if not pathlib.Path(REPO_DIR).exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "-b", BRANCH,
             f"https://github.com/{REPO}.git", REPO_DIR],
            check=True,
        )

    pathlib.Path("/app").mkdir(exist_ok=True)
    if not pathlib.Path("/app/data").exists():
        pathlib.Path("/content/data").mkdir(exist_ok=True)
        os.symlink("/content/data", "/app/data")

    os.chdir(f"{REPO_DIR}/{NOTEBOOK_REL}")

    try:
        from google.colab import userdata
        os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
        os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
    except Exception:
        print("⚠️ Set KAGGLE_USERNAME & KAGGLE_KEY via Colab Secrets (🔑 sidebar).")

    subprocess.run(["pip", "install", "-q", "kaggle"], check=True)

    wine_path = pathlib.Path("/app/data/wine-reviews/winemag-data-130k-v2.json")
    if not wine_path.exists():
        subprocess.run(
            ["kaggle", "datasets", "download", "-d", "zynicide/wine-reviews",
             "-p", "/app/data/wine-reviews", "--unzip"],
            check=True,
        )

    import tensorflow as tf
    print(f"✅ Colab setup done.\n   cwd: {os.getcwd()}\n   TF: {tf.__version__}\n   GPUs: {tf.config.list_physical_devices('GPU')}")


✅ Colab setup done.
   cwd: /content/repo/notebooks/09_transformer/gpt
   TF: 2.20.0
   GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## このノートブックの概要

ワインレビューのデータセットを使って、デコーダ型トランスフォーマ (GPT) を学習するノートブックです。書籍 9 章の流れに沿って、トークン化 → 因果アテンション → トランスフォーマブロック → 訓練 → 生成 + アテンション可視化、と段階的に組み立てていきます。

主な構成要素:

- **トークン化**: `TextVectorization` で単語列を整数 ID 列に変換
- **位置エンコーディング**: トークン埋め込み + 位置埋め込みの加算で語順情報を保持
- **因果マスク付きマルチヘッドアテンション**: 未来のトークンを参照させない自己回帰の仕組み
- **シングルブロック構成**: 原論文の 12 ブロックではなく 1 ブロックだけの簡略版

書籍の節番号との対応:

- §9.2.1-9.2.2 → 「Load the data」 / 「Tokenize」
- §9.2.3-9.2.6 → 「Causal mask」 / 「TransformerBlock」
- §9.2.7 → 「TokenAndPositionEmbedding」
- §9.2.8 → 「Train」
- §9.2.9 → 「Generate text」 (テキスト生成 + アテンション可視化)


In [56]:
# 標準ライブラリと TensorFlow/Keras を読み込む
import numpy as np
import json
import re
import string
from IPython.display import display, HTML  # アテンション可視化で HTML を表示するため

import tensorflow as tf
from tensorflow.keras import layers, models, losses, callbacks


## 0. Parameters <a name="parameters"></a>

ハイパーパラメータの意味:

| パラメータ | 意味 |
|---|---|
| `VOCAB_SIZE` | 語彙サイズ (上位 1 万語まで。それ以外は `[UNK]` トークン) |
| `MAX_LEN` | モデルに入力するトークン列の最大長 |
| `EMBEDDING_DIM` | トークン/位置埋め込みベクトルの次元 |
| `KEY_DIM` | アテンションのキー/クエリベクトルの次元 |
| `N_HEADS` | マルチヘッドアテンションのヘッド数 |
| `FEED_FORWARD_DIM` | トランスフォーマブロック内の全結合層の中間次元 |
| `BATCH_SIZE` | ミニバッチサイズ |
| `EPOCHS` | 訓練エポック数 |

`LOAD_MODEL = True` にすると、保存済みモデル (`./models/gpt.keras`) を読み込んで再学習をスキップできます。


In [57]:
VOCAB_SIZE = 10000          # 語彙サイズ (頻度上位 1 万語まで採用、それ以外は [UNK])
MAX_LEN = 80                # モデルが扱う系列長 (トークン数)
EMBEDDING_DIM = 256         # トークン/位置埋め込みの次元
KEY_DIM = 256               # アテンションのキー/クエリの次元
N_HEADS = 2                 # マルチヘッドアテンションのヘッド数
FEED_FORWARD_DIM = 256      # トランスフォーマブロック内 FFN の中間次元
VALIDATION_SPLIT = 0.2      # (このノートブックでは未使用)
SEED = 42                   # 乱数シード
LOAD_MODEL = False          # True にすると保存済みモデルを再ロードして学習をスキップ
BATCH_SIZE = 32             # ミニバッチサイズ
EPOCHS = 5                  # 訓練エポック数


## 1. Load the data <a name="load"></a>

Kaggle の `wine-reviews` データセット (約 13 万件) を読み込み、`wine review : <country> : <province> : <variety> : <description>` の形式の文字列に整形します。

冒頭に「`wine review : `」を付ける狙いは、生成時にこれをプロンプトとして与えると「ワインレビューを書く」モードでテキストが続くようにモデルを誘導できるようにすることです (書籍 9.2.9.1 で `temperature` を変えながら使う prompt と一致)。


In [58]:
# Kaggle からダウンロードしたワインレビュー JSON を読み込む (約 13 万件)
with open("/app/data/wine-reviews/winemag-data-130k-v2.json") as json_data:
    wine_data = json.load(json_data)


In [59]:
# 1 件のレビューの構造を確認 (country, province, variety, description などの dict)
wine_data[10]


{'points': '87',
 'title': 'Kirkland Signature 2011 Mountain Cuvée Cabernet Sauvignon (Napa Valley)',
 'description': 'Soft, supple plum envelopes an oaky structure in this Cabernet, supported by 15% Merlot. Coffee and chocolate complete the picture, finishing strong at the end, resulting in a value-priced wine of attractive flavor and immediate accessibility.',
 'taster_name': 'Virginie Boone',
 'taster_twitter_handle': '@vboone',
 'price': 19,
 'designation': 'Mountain Cuvée',
 'variety': 'Cabernet Sauvignon',
 'region_1': 'Napa Valley',
 'region_2': 'Napa',
 'province': 'California',
 'country': 'US',
 'winery': 'Kirkland Signature'}

In [60]:
# country / province / variety / description が揃っているレビューだけを残し、
# "wine review : <country> : <province> : <variety> : <description>" の形式に整形
filtered_data = [
    "wine review : "
    + x["country"]
    + " : "
    + x["province"]
    + " : "
    + x["variety"]
    + " : "
    + x["description"]
    for x in wine_data
    if x["country"] is not None
    and x["province"] is not None
    and x["variety"] is not None
    and x["description"] is not None
]


In [61]:
# フィルタ後の件数を確認 (NULL を含むレビューが落ちて少し減る)
n_wines = len(filtered_data)
print(f"{n_wines} recipes loaded")


129907 recipes loaded


In [62]:
# 整形済みデータの中身を 1 件覗いてみる
example = filtered_data[25]
print(example)


wine review : US : California : Pinot Noir : Oak and earth intermingle around robust aromas of wet forest floor in this vineyard-designated Pinot that hails from a high-elevation site. Small in production, it offers intense, full-bodied raspberry and blackberry steeped in smoky spice and smooth texture.


## 2. Tokenize the data <a name="tokenize"></a>

GPT は単語単位のトークンを扱うため、句読点を独立した「単語」として認識させる前処理 (`pad_punctuation`) を入れます。例えば `"great wine."` → `"great wine ."` のように記号の周りに空白を入れることで、Keras の `TextVectorization` がそれぞれ独立したトークンとして語彙に登録します。

`TextVectorization` の主な引数:

- `standardize="lower"` … 小文字化のみ (句読点は前処理済みなので除去しない)
- `max_tokens=VOCAB_SIZE` … 上位 `VOCAB_SIZE` 件だけ採用、残りは `[UNK]`
- `output_sequence_length=MAX_LEN + 1` … `+1` しているのは、後段で入力 (前 N) とターゲット (後 N) に **1 トークンずらして** 分割するため


### `pad_punctuation` の正規表現の中身

`pad_punctuation` は **記号類の前後に半角スペースを入れて、独立した単語として扱えるようにする** 関数です。2 行の `re.sub` がそれぞれ別の役割を担います。

#### 1 行目: `re.sub(f"([{string.punctuation}, '\n'])", r" \1 ", s)`

**検索パターン**: f-string で `string.punctuation` (= `!"#$%&'()*+,-./:;<=>?@[\]^_` `` ` ``{|}~`) を展開すると、最終的な正規表現はこうなります:

```
([!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~, '\n'])
```

- `[ ... ]` … **文字クラス** (「この中のどれか 1 文字にマッチ」)
- 中身は `string.punctuation` の全記号 + `,` `'` `\n` (改行)
- `(...)` で囲むと **キャプチャグループ** になり、マッチした 1 文字を `\1` で参照可能になる

**置換文字列**: `r" \1 "` … 前後に半角スペースを付けてマッチした記号を戻す。

実行例:

```
入力:  "This wine, full-bodied. Great!"
1行目: "This wine ,  full - bodied .  Great ! "
```

各記号 (`,` `-` `.` `!`) の前後にスペースが挟まりました。

#### 2 行目: `re.sub(" +", " ", s)`

`" +"` は「半角スペースが 1 個以上連続」。それを 1 個のスペースに置換 → **連続スペースを 1 個に縮める** 処理です。

1 行目で元々あったスペースと新たに足したスペースが重なって `"   "` のように複数並んでしまうので、ここで掃除します。

```
1行目後: "This wine ,  full - bodied .  Great ! "
2行目後: "This wine , full - bodied . Great ! "
```

#### なぜこの前処理が必要か

直後の `TextVectorization` 層は **「半角スペース区切りで単語に切る」** ので、前処理せずに `"wine,"` を渡すと `wine,` が 1 トークンとして語彙登録されてしまい、`wine` と別物として扱われます。前後にスペースを入れることで `wine` と `,` を独立したトークンに分けています。


In [63]:
# 記号類を独立した「単語」として扱えるよう、前後にスペースを入れる前処理
def pad_punctuation(s):
    # 1. string.punctuation の全記号 + 改行の前後に半角スペースを挟む
    s = re.sub(f"([{string.punctuation}, '\n'])", r" \1 ", s)
    # 2. 連続したスペースを 1 個に縮める
    s = re.sub(" +", " ", s)
    return s


# 全レビューに前処理を適用
text_data = [pad_punctuation(x) for x in filtered_data]


In [64]:
# 前処理後の同じレビューを確認 (句読点の前後にスペースが入っているはず)
example_data = text_data[25]
example_data


'wine review : US : California : Pinot Noir : Oak and earth intermingle around robust aromas of wet forest floor in this vineyard - designated Pinot that hails from a high - elevation site . Small in production , it offers intense , full - bodied raspberry and blackberry steeped in smoky spice and smooth texture . '

In [65]:
# TextVectorization に渡すため、tf.data.Dataset に変換しバッチ化 + シャッフル
text_ds = (
    tf.data.Dataset.from_tensor_slices(text_data)
    .batch(BATCH_SIZE)
    .shuffle(1000)
)


In [66]:
# 単語列を整数 ID 列に変換する TextVectorization 層を定義
vectorize_layer = layers.TextVectorization(
    standardize="lower",                  # 小文字化のみ (句読点は前処理済みなので除去しない)
    max_tokens=VOCAB_SIZE,                # 上位 VOCAB_SIZE 語のみ採用 (残りは [UNK])
    output_mode="int",                    # 整数 ID 列で出力
    output_sequence_length=MAX_LEN + 1,   # +1 は入力/ターゲットを 1 トークンずらして作るため
)


In [67]:
# データセットを 1 周走査して語彙を学習し、結果を取り出す
vectorize_layer.adapt(text_ds)
vocab = vectorize_layer.get_vocabulary()


In [68]:
# 語彙の先頭 10 件を確認 (0 = パディング、1 = [UNK] が予約済み)
for i, word in enumerate(vocab[:10]):
    print(f"{i}: {word}")


0: 
1: [UNK]
2: :
3: ,
4: .
5: and
6: the
7: wine
8: a
9: of


In [69]:
# 先ほどの example_data を整数 ID 列に変換した結果を確認
example_tokenised = vectorize_layer(example_data)
print(example_tokenised.numpy())


[   7   10    2   20    2   29    2   43   62    2   55    5  243 4145
  453  634   26    9  497  499  667   17   12  142   14 2214   43   25
 2484   32    8  223   14 2213  948    4  594   17  987    3   15   75
  237    3   64   14   82   97    5   74 2633   17  198   49    5  125
   77    4    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0]


## 3. Create the Training Set <a name="create"></a>

自己回帰モデルの訓練データは「**入力は最後の 1 トークンを除いた系列、ターゲットは最初の 1 トークンを除いた系列**」というシフト構造になります。

```
tokens : [t0, t1, t2, t3, t4]
x      : [t0, t1, t2, t3]
y      : [t1, t2, t3, t4]
```

各位置 `i` で「次のトークン `tokens[i+1]` を予測する」タスクに帰着するので、`fit()` 中の損失計算では位置ごとに独立に sparse cross-entropy が掛かります。


In [70]:
# 1 トークンずらして「入力 x」と「ターゲット y」を作る (自己回帰学習の標準形)
def prepare_inputs(text):
    text = tf.expand_dims(text, -1)               # 形状を整える
    tokenized_sentences = vectorize_layer(text)   # 文字列 → 整数 ID 列
    x = tokenized_sentences[:, :-1]               # 入力: 末尾を除いた系列
    y = tokenized_sentences[:, 1:]                # ターゲット: 先頭を除いた系列
    return x, y


# text_ds の各バッチに prepare_inputs を適用
train_ds = text_ds.map(prepare_inputs)


In [71]:
# train_ds から 1 バッチ取り出して中身を確認
example_input_output = train_ds.take(1).get_single_element()


In [72]:
# バッチ内 1 件目の入力 (整数 ID 列、長さ MAX_LEN=80)
example_input_output[0][0]


<tf.Tensor: shape=(80,), dtype=int64, numpy=
array([   7,   10,    2,   20,    2,  103,    2,   45,  200,    2,   12,
        411,   48,  430,    7,   32,    6, 4748,  206, 4666,   48,  205,
          5, 2981, 1620, 3006,  412,   83, 2359,  130,    3,   94,   90,
          3, 6384,    3,   36,    5,   81,   26,    4,   15,   18,   21,
        129,   82,    3,   11,  797,   34,    5,  548,   94,   16,    4,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0])>

In [73]:
# 同じ位置の出力 (入力を 1 トークン右にシフトしたもの = 次トークン)
example_input_output[1][0]


<tf.Tensor: shape=(80,), dtype=int64, numpy=
array([  10,    2,   20,    2,  103,    2,   45,  200,    2,   12,  411,
         48,  430,    7,   32,    6, 4748,  206, 4666,   48,  205,    5,
       2981, 1620, 3006,  412,   83, 2359,  130,    3,   94,   90,    3,
       6384,    3,   36,    5,   81,   26,    4,   15,   18,   21,  129,
         82,    3,   11,  797,   34,    5,  548,   94,   16,    4,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0])>

## 5. Create the causal attention mask function <a name="causal"></a>

デコーダ型トランスフォーマでは、位置 `t` のトークンが位置 `t+1, t+2, ...` を「見えない」ようにする必要があります (見えると訓練時に答えを覗いてしまい、生成時の挙動と乖離する)。

`causal_attention_mask` は下三角形の 0/1 マスクを返し、これをマルチヘッドアテンションに渡すことで、ソフトマックスを取る前に未来位置のロジットを `-inf` 相当に潰します。下のセルの転置出力は、行 `i` が「位置 `i` から見て参照できる位置 (= 列 `j ≤ i`)」を示しています (下三角に 1 が立つ)。


In [74]:
# 因果マスク: 位置 i が位置 j (j > i) を参照しないようにする下三角マスクを生成
def causal_attention_mask(batch_size, n_dest, n_src, dtype):
    i = tf.range(n_dest)[:, None]
    j = tf.range(n_src)
    m = i >= j - n_src + n_dest          # i (行) >= j (列) なら参照可能 (= 1)
    mask = tf.cast(m, dtype)
    mask = tf.reshape(mask, [1, n_dest, n_src])
    mult = tf.concat(
        [tf.expand_dims(batch_size, -1), tf.constant([1, 1], dtype=tf.int32)], 0
    )
    return tf.tile(mask, mult)           # バッチサイズ分タイル展開


# 動作確認: 10x10 のマスクを表示 (下三角に 1 が立つ)
np.transpose(causal_attention_mask(1, 10, 10, dtype=tf.int32)[0])


array([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
       [0, 1, 1, 1, 1, 1, 1, 1, 1, 1],
       [0, 0, 1, 1, 1, 1, 1, 1, 1, 1],
       [0, 0, 0, 1, 1, 1, 1, 1, 1, 1],
       [0, 0, 0, 0, 1, 1, 1, 1, 1, 1],
       [0, 0, 0, 0, 0, 1, 1, 1, 1, 1],
       [0, 0, 0, 0, 0, 0, 1, 1, 1, 1],
       [0, 0, 0, 0, 0, 0, 0, 1, 1, 1],
       [0, 0, 0, 0, 0, 0, 0, 0, 1, 1],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 1]], dtype=int32)

## 6. Create a Transformer Block layer <a name="transformer"></a>

トランスフォーマブロックは次の構造を持ちます:

```
入力 ─┬─→ MultiHeadAttention (因果マスク付き) ─→ Dropout ─┐
      │                                                       │
      └────────────────── 残差 + LayerNorm ←──────────────────┘
                              │
                              ├─→ Dense(FFN_DIM, relu) → Dense(EMBED_DIM) → Dropout ─┐
                              │                                                          │
                              └──────────── 残差 + LayerNorm ←──────────────────────────┘
                                                  │
                                                  ↓
                                              出力 + アテンションスコア
```

`return_attention_scores=True` でアテンション重みも返しているのは、後で 9.2.9.2 (アテンション可視化) で使うためです。`get_config()` を実装しているのは、モデルを `.keras` 形式で保存→ロードしたときにこのカスタム層を再構築できるようにするためです。


In [75]:
# マルチヘッドアテンション + FFN + 残差接続 + LayerNorm を 1 つにまとめた層
class TransformerBlock(layers.Layer):
    def __init__(self, num_heads, key_dim, embed_dim, ff_dim, dropout_rate=0.1):
        super(TransformerBlock, self).__init__()
        self.num_heads = num_heads
        self.key_dim = key_dim
        self.embed_dim = embed_dim
        self.ff_dim = ff_dim
        self.dropout_rate = dropout_rate
        # 1. マルチヘッドアテンション層 (因果マスクで未来トークンを隠す)
        self.attn = layers.MultiHeadAttention(
            num_heads, key_dim, output_shape=embed_dim
        )
        self.dropout_1 = layers.Dropout(self.dropout_rate)
        self.ln_1 = layers.LayerNormalization(epsilon=1e-6)
        # 2. フィードフォワード (位置ごとの 2 層 MLP)
        self.ffn_1 = layers.Dense(self.ff_dim, activation="relu")
        self.ffn_2 = layers.Dense(self.embed_dim)
        self.dropout_2 = layers.Dropout(self.dropout_rate)
        self.ln_2 = layers.LayerNormalization(epsilon=1e-6)

    def call(self, inputs):
        input_shape = tf.shape(inputs)
        batch_size = input_shape[0]
        seq_len = input_shape[1]
        # 各バッチ・各系列長に合わせて因果マスクを生成
        causal_mask = causal_attention_mask(
            batch_size, seq_len, seq_len, tf.bool
        )
        # Self-attention: query = key = value = inputs (自己参照型)
        attention_output, attention_scores = self.attn(
            inputs,
            inputs,
            attention_mask=causal_mask,
            return_attention_scores=True,   # 可視化用にアテンションスコアも返す
        )
        attention_output = self.dropout_1(attention_output)
        out1 = self.ln_1(inputs + attention_output)   # 残差接続 + LayerNorm
        # フィードフォワード経路
        ffn_1 = self.ffn_1(out1)
        ffn_2 = self.ffn_2(ffn_1)
        ffn_output = self.dropout_2(ffn_2)
        # もう一度残差接続 + LayerNorm
        return (self.ln_2(out1 + ffn_output), attention_scores)

    def get_config(self):
        # モデル保存/ロード時にカスタム層を再構築するためのコンフィグ
        config = super().get_config()
        config.update(
            {
                "key_dim": self.key_dim,
                "embed_dim": self.embed_dim,
                "num_heads": self.num_heads,
                "ff_dim": self.ff_dim,
                "dropout_rate": self.dropout_rate,
            }
        )
        return config


## 7. Create the Token and Position Embedding <a name="embedder"></a>

アテンションは順序を意識しません (キーとクエリのドット積はすべての位置ペアで並列に計算されるため)。例えば次の 2 文はアテンション層から見ると区別できません:

- *The dog looked at the boy* … (吠えた?)
- *The boy looked at the dog* … (微笑んだ?)

この問題を解消するため、トークン埋め込みに **位置埋め込み** を加算します。

```
最終的な埋め込み = TokenEmbedding(token_id) + PositionEmbedding(position)
```

原論文 (Vaswani et al., 2017) では sin/cos の三角関数による固定位置エンコーディングが使われましたが、GPT では学習可能な `Embedding` 層を使うのが標準です (このノートブックもそれに従う)。


In [76]:
# トークン埋め込み + 位置埋め込みの加算層
class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, max_len, vocab_size, embed_dim):
        super(TokenAndPositionEmbedding, self).__init__()
        self.max_len = max_len
        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        # トークン ID → ベクトル
        self.token_emb = layers.Embedding(
            input_dim=vocab_size, output_dim=embed_dim
        )
        # 位置 (0, 1, 2, ..., max_len-1) → ベクトル
        self.pos_emb = layers.Embedding(input_dim=max_len, output_dim=embed_dim)

    def call(self, x):
        maxlen = tf.shape(x)[-1]
        positions = tf.range(start=0, limit=maxlen, delta=1)
        positions = self.pos_emb(positions)
        x = self.token_emb(x)
        return x + positions   # 加算するだけ (連結ではない)

    def get_config(self):
        config = super().get_config()
        config.update(
            {
                "max_len": self.max_len,
                "vocab_size": self.vocab_size,
                "embed_dim": self.embed_dim,
            }
        )
        return config


## 8. Build the Transformer model <a name="transformer_decoder"></a>

これまで定義した部品を組み立てます (書籍の図 9-9 に相当):

```
Input
  ↓
TokenAndPositionEmbedding
  ↓
TransformerBlock  ← 本来は 12 層積むが、簡略化のため 1 層のみ
  ↓
Dense(VOCAB_SIZE, softmax)  ← 次トークンの確率分布
```

モデルは出力を 2 つ持ちます: 語彙分布 (`outputs`) と TransformerBlock からのアテンションスコア (`attention_scores`)。`compile()` で渡す `loss=[..., None]` の 2 つ目が `None` になっているのは、アテンションスコア側には損失をかけないという指定です (生成時の可視化用に取り出すだけ)。


In [77]:
# GPT モデル全体を組み立てる
inputs = layers.Input(shape=(None,), dtype=tf.int32)                          # 可変長の整数 ID 列
x = TokenAndPositionEmbedding(MAX_LEN, VOCAB_SIZE, EMBEDDING_DIM)(inputs)     # 埋め込み (トークン + 位置)
x, attention_scores = TransformerBlock(                                       # トランスフォーマブロック (1 個だけ)
    N_HEADS, KEY_DIM, EMBEDDING_DIM, FEED_FORWARD_DIM
)(x)
outputs = layers.Dense(VOCAB_SIZE, activation="softmax")(x)                   # 各位置で次トークンの確率分布
gpt = models.Model(inputs=inputs, outputs=[outputs, attention_scores])         # 2 出力 (語彙分布 + アテンション)
# 損失は語彙分布側にだけかける (アテンション側は None)
gpt.compile("adam", loss=[losses.SparseCategoricalCrossentropy(), None])


In [78]:
gpt.summary()   # モデル構造とパラメータ数を表示


Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_2 (InputLayer)      │ (None, None)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ token_and_position_embedding_2  │ (None, None, 256)      │     2,580,480 │
│ (TokenAndPositionEmbedding)     │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block_2             │ [(None, None, 256),    │       658,688 │
│ (TransformerBlock)              │ (None, 2, None, None)] │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, None, 10000)    │     2,570,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,809,168 (22.16 MB)

 Trainable params: 5,809,168 (22.16 MB)

 Non-trainable params: 0 (0.00 B)

In [79]:
# LOAD_MODEL=True なら保存済みモデルを再ロードして学習をスキップ
if LOAD_MODEL:
    # model.load_weights('./models/model')
    gpt = models.load_model("./models/gpt.keras", compile=True)


## 9. Train the Transformer <a name="train"></a>

学習時のセットアップとして、3 つのコールバックを使います:

1. **`TextGenerator`** … 自前のカスタム `callbacks.Callback`。毎エポック終了時に「wine review」というプロンプトから 80 トークンを生成してログ表示する。学習が進むにつれて出力がどう改善されるかを観察できる
2. **`ModelCheckpoint`** … エポックごとに重みを `./checkpoint/checkpoint.weights.h5` に保存。途中で中断しても再開できる
3. **`TensorBoard`** … `./logs` 配下に学習ログを出力。Colab 上で `%load_ext tensorboard` & `%tensorboard --logdir ./logs` で可視化可能

`TextGenerator.generate()` の `sample_from(probs, temperature)` は、`p ← p^(1/T) / Σp^(1/T)` で確率分布を「鋭く/平坦に」してからサンプリングします (`T < 1` で鋭く決定論的に、`T > 1` で平坦で多様に)。


In [80]:
# エポックごとにテキスト生成テストを行うカスタムコールバック
class TextGenerator(callbacks.Callback):
    def __init__(self, index_to_word, top_k=10):
        self.index_to_word = index_to_word   # ID → 単語の対応 (語彙リスト)
        self.word_to_index = {                # 逆方向: 単語 → ID
            word: index for index, word in enumerate(index_to_word)
        }

    def sample_from(self, probs, temperature):
        # temperature サンプリング: p ← p^(1/T) / Σp^(1/T)
        # T < 1 で分布が鋭く (決定論的)、T > 1 で平坦 (多様) になる
        probs = probs ** (1 / temperature)
        probs = probs / np.sum(probs)
        return np.random.choice(len(probs), p=probs), probs

    def generate(self, start_prompt, max_tokens, temperature):
        # プロンプトをトークン ID 列に変換 (未知語は 1 = [UNK])
        start_tokens = [
            self.word_to_index.get(x, 1) for x in start_prompt.split()
        ]
        sample_token = None
        info = []   # 各ステップのプロンプト/確率/アテンションを記録
        # max_tokens に達するか、終端トークン (0) が出るまで生成
        while len(start_tokens) < max_tokens and sample_token != 0:
            x = np.array([start_tokens])
            y, att = self.model.predict(x, verbose=0)
            # 最後の位置の出力分布から次トークンをサンプリング
            sample_token, probs = self.sample_from(y[0][-1], temperature)
            info.append(
                {
                    "prompt": start_prompt,
                    "word_probs": probs,
                    "atts": att[0, :, -1, :],   # 最後の位置のアテンション
                }
            )
            start_tokens.append(sample_token)
            start_prompt = start_prompt + " " + self.index_to_word[sample_token]
        print(f"\ngenerated text:\n{start_prompt}\n")
        return info

    def on_epoch_end(self, epoch, logs=None):
        # 各エポック末に「wine review」プロンプトで生成して進捗を観察
        self.generate("wine review", max_tokens=80, temperature=1.0)


In [81]:
# エポックごとに重みを保存するチェックポイント
model_checkpoint_callback = callbacks.ModelCheckpoint(
    filepath="./checkpoint/checkpoint.weights.h5",
    save_weights_only=True,
    save_freq="epoch",
    verbose=0,
)

# TensorBoard 用のログ出力
tensorboard_callback = callbacks.TensorBoard(log_dir="./logs")

# 訓練中の生成テストを行うコールバック
text_generator = TextGenerator(vocab)


In [82]:
# 学習開始 (Colab T4 GPU で 1 エポックあたり数分)
gpt.fit(
    train_ds,
    epochs=EPOCHS,
    callbacks=[model_checkpoint_callback, tensorboard_callback, text_generator],
)


Epoch 1/5
4059/4060 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - loss: 2.6018
generated text:
wine review : spain : northern spain : tempranillo : blueberry , plum , plum and char are potent and vanilla aromas lead to creamy and not semi - dry surface . this feels plush on the palate , with hard - driving plum and berry flavors . this type of slightly oaky wine with more structure and some of the almost nutty finish that ' s feeling bullish and not overpowering . drink through 2023 . 

4060/4060 ━━━━━━━━━━━━━━━━━━━━ 190s 45ms/step - loss: 2.2466
Epoch 2/5
4059/4060 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 1.9824
generated text:
wine review : us : washington : cabernet sauvignon : a beautifully floral aroma combine to provide seductive layers of gun lemon and black cherry , followed by finished beautifully balanced layers of richness . gulp down easy to drink here . 

4060/4060 ━━━━━━━━━━━━━━━━━━━━ 113s 28ms/step - loss: 1.9587
Epoch 3/5
4059/4060 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - loss: 1.899

In [83]:
# 学習完了後のモデル全体を Keras 3 形式 (.keras) で保存
gpt.save("./models/gpt.keras")


# 3. Generate text using the Transformer

**書籍 §9.2.9「GPT の分析」に対応するセクション**です。訓練済みモデルを使って 2 つの分析を行います:

### §9.2.9.1 テキスト生成

`temperature` パラメータの効果を観察します:

- `temperature = 1.0` … 分布のままサンプリングする (多様だが時々おかしい単語が出る)
- `temperature = 0.5` … 分布を鋭くしてからサンプリング (決定論的に近く、最頻語に偏る → 自然な文だが画一的)

### §9.2.9.2 アテンション可視化

`print_probs(info, vocab)` が以下を出力します:

- **HTML ハイライト**: アテンション重みの平均を背景色 (青の濃淡) で表示。「ドイツ → mosel/rheingau」のように、関連する位置にどれだけ注意が向いているかが視覚的にわかる
- **トップ K 単語の確率分布**: 各位置でモデルが次に何を予測しようとしているかを上位 5 件で表示 (図 9-11 と同じ)

入力プロンプトの言葉を変えて、モデルが文中のどの単語に注目するかを観察してみてください。


In [84]:
# 生成過程の info を可視化する関数
def print_probs(info, vocab, top_k=5):
    for i in info:
        # 1. アテンション重みを背景色でハイライトした HTML を生成
        highlighted_text = []
        for word, att_score in zip(
            i["prompt"].split(), np.mean(i["atts"], axis=0)   # ヘッド方向に平均
        ):
            highlighted_text.append(
                '<span style="background-color:rgba(135,206,250,'
                + str(att_score / max(np.mean(i["atts"], axis=0)))   # 0〜1 に正規化
                + ');">'
                + word
                + "</span>"
            )
        highlighted_text = " ".join(highlighted_text)
        display(HTML(highlighted_text))

        # 2. 次トークン候補のトップ K を確率付きで表示
        word_probs = i["word_probs"]
        p_sorted = np.sort(word_probs)[::-1][:top_k]
        i_sorted = np.argsort(word_probs)[::-1][:top_k]
        for p, i in zip(p_sorted, i_sorted):
            print(f"{vocab[i]}:   \t{np.round(100*p,2)}%")
        print("--------\n")


In [85]:
# 米国産ワインのレビューを temperature=1.0 (多様) で生成
info = text_generator.generate(
    "wine review : us", max_tokens=80, temperature=1.0
)



generated text:
wine review : us : california : zinfandel : the alcohol is a touch hot in alcohol . with glyceriney sweetness , but this wine is unbalanced , a disappointing wine . it was thin , with watered - down favor cherry juice , but it does contain a bitter touch of chocolate . 



In [86]:
# イタリア産ワインを temperature=0.5 (決定論的寄り) で生成
info = text_generator.generate(
    "wine review : italy", max_tokens=80, temperature=0.5
)



generated text:
wine review : italy : tuscany : sangiovese grosso : this is a beautifully shaped wine with a good intensity of fruit and a layer of spice and tobacco . the wine is full of ripe fruit and exotic spice notes that are backed by firm tannins and bold flavors of black cherry and cassis . drink after 2014 . 



In [87]:
# ドイツ産ワインを生成し、各位置のアテンション + 確率分布を可視化 (図 9-11 相当)
info = text_generator.generate(
    "wine review : germany", max_tokens=80, temperature=0.5
)
print_probs(info, vocab)



generated text:
wine review : germany : mosel : riesling : this gorgeously aromatic , with notes of white blossoms , honey and saffron . it ' s rich in flavor , yet deeply concentrated and penetrating , it ' s intensely concentrated with sweet honey and marmalade flavors , but it ' s gorgeously complex , with a long , meandering finish . 



::   	100.0%
-:   	0.0%
grosso:   	0.0%
grasparossa:   	0.0%
review:   	0.0%
--------



mosel:   	60.38999938964844%
rheinhessen:   	19.440000534057617%
rheingau:   	8.6899995803833%
pfalz:   	6.210000038146973%
franken:   	3.7799999713897705%
--------



::   	99.9800033569336%
-:   	0.019999999552965164%
laurent:   	0.0%
weissburgunder:   	0.0%
s:   	0.0%
--------



riesling:   	99.9800033569336%
pinot:   	0.019999999552965164%
spätburgunder:   	0.0%
chardonnay:   	0.0%
dornfelder:   	0.0%
--------



...

,:   	99.20999908447266%
finish:   	0.4300000071525574%
yet:   	0.1899999976158142%
-:   	0.03999999910593033%
and:   	0.03999999910593033%
--------



meandering:   	38.470001220703125%
lingering:   	23.950000762939453%
honeyed:   	13.989999771118164%
penetrating:   	4.320000171661377%
mineral:   	4.210000038146973%
--------



finish:   	96.30999755859375%
veins:   	1.600000023841858%
marmalade:   	0.46000000834465027%
notes:   	0.27000001072883606%
honey:   	0.1899999976158142%
--------



.:   	90.31999969482422%
that:   	7.360000133514404%
marked:   	1.0099999904632568%
of:   	0.6100000143051147%
lingering:   	0.12999999523162842%
--------



:   	96.51000213623047%
drink:   	1.9700000286102295%
it:   	1.100000023841858%
the:   	0.11999999731779099%
a:   	0.07999999821186066%
--------

